# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haiqa037/Repository_ML_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Each row represents a content page (content_id), summarized over the last 90 days leading up to the export date of the initial CSV. This is a one-time snapshot, not a time series — each page is shown once, already summed up, not once daily.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Total rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Rows == unique content_id? {len(df) == df['content_id'].nunique()}")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Attributes: impressions_90d, clicks_90d, sessions_90d, avg_position, ctr, word_count, content_age_days, freshness tier, content_type, primary_intent, competition level, ai_sessions_90d, ai_traffic_pct.

Label: trend_direction (specifically is_declining_label = trend_direction == "down") — a surrogate label derived from the present window, not a prospective result.

Context: client_id (used solely for grouping/holdout purposes, never as a predictive element — doing so would allow the model to recall particular clients rather than understanding overarching trends).

Excluded: any score calculated for products such as health_score or priority_score — these are not included in this dataset, but if I were to create one in the future, it would be intentionally excluded from modeling, as training on a product's own decision simply instructs the model to replicate that decision rather than uncover new insights.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.columns.tolist())
print(f"\nMissing values per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Each claim in Sections 1-2 requires a supporting query  grain , field comprehensiveness, and whether the label is truly sourced from the feature window without disclosing future data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain check
assert len(df) == df['content_id'].nunique(), "Grain violated: duplicate content_id rows exist"

# Filtered population, matching the starter pipeline's own filter rule
df_clean = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")
print(f"Rows after impressions_90d > 0 and content_age_days >= 90 filter: {len(df_clean):,}")

# Label distribution
print(df_clean["trend_direction"].value_counts())

# Confirm label window doesn't reference anything beyond the 90d feature window
print(f"\nColumns referencing future-looking data: "
      f"{[c for c in df.columns if 'future' in c.lower() or 'next' in c.lower()]}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This initial CSV represents a small (~30k row) anonymized single-snapshot extract, not the complete warehouse therefore, it cannot illustrate true per-client historical depth, does not differentiate rows prior to a client's GA4 tracking initiation (which the entire warehouse indicates with ga4_data_available = FALSE), and lacks daily time series data to assess seasonality or consistency. Since it’s a single 90-day period rather than daily information, I cannot confirm that my "feature window" and "target window" do not coincide  that verification can only occur once I access the warehouse's daily fact table for subsequent weeks. Currently, every claim  drawn from this data is a momentary correlation, not a trend I have confirmed continues over time.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# What this dataset structurally cannot show: no report_date / daily grain column exists
date_like_cols = [c for c in df.columns if 'date' in c.lower() or 'day' in c.lower()]
print(f"Date/day-related columns available: {date_like_cols}")
print("No daily time-series column present — confirms single-snapshot limitation described above.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.